# Customising Existing Indicators with Registry and YAML

When calculating climate indices, you often need to adapt an existing scientific indicator rather than implementing one from scratch—for example, changing a temperature threshold (such as 30°C for heat extremes) or fixing the aggregation frequency to annual (`YS`).

**earthkit-climate** supports two complementary ways to achieve this without writing new calculation logic:

1. **Reparameterising via `registry`**: Programmatically configure an existing `xclim` indicator and wrap it with `wrap_indicator`.
2. **Declarative Virtual Modules via YAML**: Define indicators in a YAML configuration file and compile them into virtual Python modules with `build_indicator_module_from_yaml`.

This guide demonstrates both solutions on sample daily maximum temperature data and verifies that both produce identical results.

## Solution 1: Reparameterising via the registry

You can look up any base indicator in `xclim.core.indicator.registry` (such as `TX_DAYS_ABOVE`), customize its default parameters (such as `thresh="30 degC"` and `freq="YS"`), and wrap it with `wrap_indicator` so that it seamlessly accepts `earthkit-data` objects and `xarray` datasets:

In [1]:
import earthkit.data as ekd
from xclim.core.indicator import registry

from earthkit.climate.utils.decorators import wrap_indicator

# Load sample maximum temperature data (33 years from 1991 to 2023)
tasmax = ekd.from_source("earthkit-climate-sample", "synthetic-daily-temperature").to_xarray()

# Derive a custom indicator fixing threshold to 30 degC and frequency to YS:
hot_days_reg = wrap_indicator(
    registry["TX_DAYS_ABOVE"](
        identifier="hot_days",
        title="Hot Days above 30°C",
        parameters=dict(thresh="30 degC", freq="YS"),
    )
)

res_reg = hot_days_reg(tasmax=tasmax)
res_reg

/home/cuadradot/predictia-projects/git/c3s-indices/earthkit-climate/src/earthkit/climate/_sample_source.py:320: UserWarning: earthkit-climate-sample datasets are made available for demonstration purposes only. Files are not guaranteed to be available long-term and may change over time. Please use official channels to obtain the contained datasets reliably for other purposes.
  warnings.warn(
/home/cuadradot/predictia-projects/git/c3s-indices/earthkit-climate/.venv/lib/python3.12/site-packages/xclim/core/indicator.py:296: UserWarning: Class HOT_DAYS already exists and will be overwritten.
  warnings.warn(f"Class {name} already exists and will be overwritten.", stacklevel=1)


<xarray.DataArray 'hot_days' (time: 33)> Size: 264B
array([ 2.,  8.,  9.,  7.,  3.,  5., 10.,  8.,  9.,  2.,  5.,  6.,  5.,
        7.,  9.,  7.,  7., 12.,  7., 11.,  2.,  4.,  7.,  8., 11.,  9.,
        8.,  5.,  8.,  7., 14.,  3.,  6.])
Coordinates:
  * time     (time) datetime64[us] 264B 1991-01-01 1992-01-01 ... 2023-01-01
Attributes:
    units:          days
    standard_name:  number_of_days_with_air_temperature_above_threshold
    cell_methods:   time: maximum time: sum over days
    history:        [2026-10-23 09:57:22] hot_days: HOT_DAYS(tasmax=tasmax, o...
    long_name:      The number of days with maximum temperature above 30 degc
    description:    Annual number of days where daily maximum temperature exc...

## Solution 2: Declarative Virtual Modules via YAML

For operational workflows (such as C3S thematic indicators), indicators can be declared in a YAML configuration file and loaded as virtual modules using `build_indicator_module_from_yaml`. The resulting indicators are then wrapped with `wrap_indicator`:

In [2]:
import tempfile

from xclim.core.indicator import build_indicator_module_from_yaml

from earthkit.climate.utils.decorators import wrap_indicator

yaml_definition = """
realm: atmos
indicators:
  hot_days:
    base: tx_days_above
    parameters:
      thresh: 30 degC
      freq: YS
    title: Hot Days above 30°C
    abstract: Annual count of days where daily maximum temperature exceeds 30°C.
"""

with tempfile.NamedTemporaryFile("w", suffix=".yml", delete=True) as f:
    f.write(yaml_definition)
    f.flush()
    sector_module = build_indicator_module_from_yaml(f.name, name="sector_heat")
    hot_days_yaml = wrap_indicator(sector_module.hot_days)

res_yaml = hot_days_yaml(tasmax=tasmax)
res_yaml

<xarray.DataArray 'hot_days' (time: 33)> Size: 264B
array([ 2.,  8.,  9.,  7.,  3.,  5., 10.,  8.,  9.,  2.,  5.,  6.,  5.,
        7.,  9.,  7.,  7., 12.,  7., 11.,  2.,  4.,  7.,  8., 11.,  9.,
        8.,  5.,  8.,  7., 14.,  3.,  6.])
Coordinates:
  * time     (time) datetime64[us] 264B 1991-01-01 1992-01-01 ... 2023-01-01
Attributes:
    units:          days
    standard_name:  number_of_days_with_air_temperature_above_threshold
    cell_methods:   time: maximum time: sum over days
    history:        [2026-10-23 09:57:22] hot_days: sector_heat.HOT_DAYS(tasm...
    long_name:      The number of days with maximum temperature above 30 degc
    description:    Annual number of days where daily maximum temperature exc...

## Verification and comparison

Both approaches define the identical scientific calculation through different interfaces (programmatic vs declarative). We can verify that both outputs are numerically identical:

In [3]:
import xarray as xr

# Verify that both approaches produce identical numerical results
assert res_reg.equals(res_yaml)

xr.Dataset({
    "registry_reparameterised": res_reg,
    "yaml_virtual_module": res_yaml,
}).to_dataframe()

,registry_reparameterised,yaml_virtual_module
time,,
1991-01-01,2.0,2.0
1992-01-01,8.0,8.0
1993-01-01,9.0,9.0
1994-01-01,7.0,7.0
1995-01-01,3.0,3.0
1996-01-01,5.0,5.0
1997-01-01,10.0,10.0
1998-01-01,8.0,8.0
1999-01-01,9.0,9.0
